# 06 — Verification: draft, critique, revise

**What you'll learn**

- Why a first answer is a *draft*: verification is a separate step that catches what generation missed
- The cheap deterministic verifier — `check_decision` recomputes the gold answer from `rules.decide` and flags any field the agent contradicts, with no model call
- The open-ended verifier — build an LLM judge step by step (a rubric, a JSON verdict, junk-proof parsing), then import `shoplab.verify.judge`
- The evaluator-optimizer loop — `refine_until(draft_fn, critique_fn)`: draft, critique, revise until a verdict passes, in a bounded number of rounds
- When verification pays for itself: microseconds-and-free deterministic checks versus a per-token judge call, and why you verify the risky slice, not everything

*Time: ~2 min on a first live run; under a minute cached. Cost: ~$0.001. Cached reruns are free.*

## A first answer is a draft

Everything so far has trusted the agent's first answer. Chapter 04 measured how often that first answer is right and found real gaps — wrong policy citations, arithmetic off by a restocking fee. The fix is not a better prompt; it is a second step. Generation and verification are different jobs: writing an answer and checking an answer draw on different work, and folding them into one pass lets the model grade its own homework in the same breath it wrote it. Split them.

The shape is the *evaluator-optimizer* loop: a drafter proposes, a critic judges, and if the critic rejects, the drafter revises using the critique. *Self-Refine: Iterative Refinement with Self-Feedback* (Madaan et al., 2023, [arXiv:2303.17651](https://arxiv.org/abs/2303.17651)) showed a single model lifting its own output this way, round after round, with no new training. The critic can be a model too — grading open-ended text against a rubric, the pattern named and measured in *Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena* (Zheng et al., 2023, [arXiv:2306.05685](https://arxiv.org/abs/2306.05685)) — or, when the task has a spec, plain code that recomputes the truth.

```text
      +---------+     draft      +----------+
      |  DRAFT  |  ----------->  | CRITIQUE |
      | (model) |                | (verify) |
      +---------+                +----+-----+
           ^                          |
           |        feedback          | passes?
           +----------- no -----------+
                                      | yes
                                      v
                                    SHIP
```

This chapter builds all three pieces of that loop for the ops desk — a deterministic critic, a model critic, and the loop that drives them — as `shoplab.verify`.

In [ ]:
# === config (identical in every notebook) ===
import os, getpass
import litellm
from dotenv import load_dotenv              # pip install -e ".[obs]" if this fails

load_dotenv(".env")   # reads OPENROUTER_API_KEY / MODEL / STRONG_MODEL (see .env.example)

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")

MODEL = os.environ.get("MODEL", "openrouter/deepseek/deepseek-v3.2")
STRONG_MODEL = os.environ.get("STRONG_MODEL", "openrouter/deepseek/deepseek-v4-flash")

# Per-notebook override: uncomment to ignore .env here (any LiteLLM provider works).
# MODEL = "openrouter/google/gemini-2.5-flash-lite"
# MODEL = "openai/gpt-4o-mini"              # direct OpenAI, uses OPENAI_API_KEY instead

TEMPERATURE = 0                             # the whole course runs at temperature 0
litellm.drop_params = True                  # ignore params a provider does not support
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=".litellm_cache")  # reruns are ~free

### Phoenix observability (optional)

The frozen cell finds or starts the local Phoenix server and traces every LiteLLM call this notebook makes — the judge calls and the two refinement loops below. Watching a `refine_until` run as a short chain of grader calls is a useful second view of the loop. Optional: skip it and nothing else changes.

In [ ]:
# optional: Phoenix tracing (see notebook 03)
import obs

obs.enable_phoenix()

## Two verifiers: recompute, or judge

Verification splits by what kind of answer you are checking. When the task has a written spec, the strongest verifier is not a model at all — it recomputes the correct answer and compares. Ticket triage has exactly that: `shoplab.rules.decide`, the cascade from chapter 04, is the spec, so a verifier can recompute the gold decision for any ticket and flag every field the agent's answer contradicts. No prompt, no model call, no judgment — the last free line of defense before a refund ships. That verifier is `shoplab.verify.check_decision`, and because it needs nothing chapter 04 did not already ship, we import it and put it straight to work.

The headline test: take a first draft the way an over-eager agent might produce one — decided from the ticket text alone, no lookups — and run it past the check before anything moves.

In [ ]:
import shoplab.llm, shoplab.verify
from shoplab.world import load_orders, load_customers, load_tickets

orders = {o["order_id"]: o for o in load_orders()}
customers = {c["customer_id"]: c for c in load_customers()}
tickets = load_tickets()

t = next(x for s in tickets.values() for x in s if x["ticket_id"] == "TKT-2205")
order, customer = orders[t["order_id"]], customers[t["customer_id"]]

def describe(t):
    return (f"sku {t['sku']}, qty {t['qty']}, condition {t['item_condition']}, "
            f"{t['days_since_delivery']} days since delivery, requested "
            f"{t['requested_action']}. Customer: {t['reason_text']}")

In [ ]:
draft = shoplab.llm.parse_json_loose(shoplab.llm.llm(
    "Decide this return ticket from its text alone, no lookups. Reply as JSON "
    '{"decision", "policy_id", "refund_usd"}. ' + describe(t), model=MODEL))

verdict = shoplab.verify.check_decision(draft, t, order, customer)
print("draft: ", draft)
print("agrees:", verdict["agrees"], " mismatch:", verdict["mismatch"])
print("gold:  ", verdict["gold"])
print("gold pred agrees:",
      shoplab.verify.check_decision(t["gold"], t, order, customer)["agrees"])

> **What you should see:** the lazy draft is wrong. Deciding from the text alone, the model misreads an opened, in-window return and lands on the wrong decision, an invented `policy_id` (often the sku echoed straight back), and an amount that is not 170.99. `check_decision` returns `agrees=False` and names every field that disagrees, with the recomputed gold in hand — and it did that with zero model calls. That is a wrong refund caught on the desk instead of in the ledger. The gold prediction, fed to the same check, returns `agrees=True` with an empty `mismatch`.

## An LLM judge for the open-ended part

Recomputation only works where a spec exists. The refund *decision* has one; the *justification the agent writes for the customer* does not — "is this explanation faithful to the policy it cites?" has no closed form. That is where the second verifier earns its place: a stronger model reads the answer against a rubric and returns a verdict. The rubric is the whole contract — vague rubrics get vague grades — and the verdict must be structured so code can branch on it. Build it in three moves: a grading system prompt, a JSON verdict, and parsing that never crashes on a model that ignores the format.

In [ ]:
import shoplab.llm

JUDGE_SYSTEM = ("You grade strictly. Given a question, an answer, and a rubric, reply with "
                'one JSON object only: {"pass": true|false, "score": 0.0-1.0, "reasons": str}.')

def judge_mini(question, answer, rubric, *, model=STRONG_MODEL):
    prompt = f"Q:\n{question}\n\nANSWER:\n{answer}\n\nRUBRIC:\n{rubric}\n\nVerdict JSON:"
    reply = shoplab.llm.llm(prompt, system=JUDGE_SYSTEM, model=model)
    try:
        v = shoplab.llm.parse_json_loose(reply)
        return {"pass": bool(v.get("pass")), "score": float(v.get("score", 0) or 0),
                "reasons": str(v.get("reasons", ""))}
    except (ValueError, TypeError, AttributeError) as exc:
        return {"pass": False, "score": 0.0, "reasons": f"unparseable: {exc}"}

In [ ]:
samples = ['{"pass": true, "score": 0.9, "reasons": "cites the policy and the math"}',
           '```json\n{"pass": false, "score": 0.2, "reasons": "no numbers"}\n```',
           "I think it reads fine, but there is no JSON here at all."]
for s in samples:
    try:
        print("parsed    ->", shoplab.llm.parse_json_loose(s))
    except ValueError as exc:
        print("fail-safe ->", {"pass": False, "score": 0.0, "reasons": f"unparseable: {exc}"})

> **What you should see:** valid JSON parses; a Markdown-fenced JSON block parses too (the loose parser strips the fence); and a prose reply with no JSON raises `ValueError`, which the judge turns into a safe `pass=False` verdict instead of a crash. A verifier that dies on bad input is not a verifier.

That three-move shape — grade against a rubric, return JSON, fail safe — is `shoplab.verify.judge`, which defaults to `STRONG_MODEL` so the grader outranks the model under test. Import it and point it at two justifications for the same ticket: one that applies the restocking policy correctly, one that waves it away.

In [ ]:
from shoplab.verify import judge

Q = "Is this justification consistent with the policy it cites?"
RUBRIC = ("Pass only if the justification correctly applies pol-restocking to an opened, "
          "non-vip, in-window item: a partial refund of 90% of item value, citing "
          "pol-restocking as its basis.")
faithful = ("The boots were opened and returned within the window by a non-vip "
            "customer, so pol-restocking applies: refund 90% of $189.99, i.e. $170.99.")
sloppy = "The customer is unhappy, so issue a full refund of $189.99 under pol-returns."

for label, ans in [("faithful", faithful), ("sloppy", sloppy)]:
    v = judge(Q, ans, RUBRIC)
    print(f"{label:9} -> pass={v['pass']} score={v['score']} | {v['reasons'][:70]}")

> **What you should see:** the faithful justification passes with a high score; the sloppy one fails, and its `reasons` string names the error — a full refund where the restocking fee was due, the wrong policy cited. The judge is doing real discrimination here, not rubber-stamping. Keep the honest caveat in view: it is a model, so the same pair on a rerun can shift a score or reword a reason, and — as the loop below will show — a confidently worded answer can talk it into a pass.

## Draft, critique, revise

Two verifiers, and neither has changed the answer yet — they only judge it. The loop closes when a failed verdict feeds back into a new draft. That is the whole evaluator-optimizer pattern, and written out it is about eight lines: draft once with no feedback, critique, and while the critique fails, draft again with the critique in hand — capped, so a drafter that never satisfies the critic still terminates.

In [ ]:
def refine_until(draft_fn, critique_fn, *, max_rounds=3):
    draft = draft_fn(None)
    history, rounds = [], 0
    for rounds in range(1, max_rounds + 1):
        verdict = critique_fn(draft)
        history.append(verdict)
        if verdict.get("pass"):
            break
        draft = draft_fn(verdict)
    return {"result": draft, "rounds": rounds, "history": history}

The contract is deliberately small. `draft_fn(feedback)` gets `None` on the first call and the previous verdict dict afterward, so the drafter can read the critique and act on it. `critique_fn(draft)` returns a verdict whose truthy `pass` stops the loop. That is exactly `shoplab.verify.refine_until`, verbatim — import it and wire in our two functions: a `draft_fn` that decides the ticket from its text, and a `critique_fn` that wraps `check_decision`. When the deterministic check rejects a draft it hands back the exact fields it got wrong, so the revision has somewhere to go.

In [ ]:
from shoplab.verify import refine_until

DECISIONS = "approve_refund|partial_refund|replacement|store_credit|deny|escalate"

def draft_decision(feedback):
    msg = ('Decide this return ticket and justify it in one sentence. Reply as JSON '
           '{"decision", "policy_id", "refund_usd" (number or null), "justification"}, '
           'with decision one of ' + DECISIONS + '. ' + describe(t))
    if feedback is not None:
        g = feedback["gold"]
        msg += (" A policy check REJECTED your draft; the correct fields are "
                f"decision={g['decision']}, policy_id={g['policy_id']}, "
                f"refund_usd={g['refund_usd']}. Return those exact fields with a matching justification.")
    return shoplab.llm.parse_json_loose(shoplab.llm.llm(msg, model=MODEL))

def check_critic(draft):
    v = shoplab.verify.check_decision(draft, t, order, customer)
    return {"pass": v["agrees"], "mismatch": v["mismatch"], "gold": v["gold"]}

out = refine_until(draft_decision, check_critic, max_rounds=3)
print("stopped after", out["rounds"], "round(s)")
for i, v in enumerate(out["history"], 1):
    print(f"  round {i}: pass={v['pass']} mismatch={v['mismatch']}")
print("final:", out["result"])

> **What you should see:** the first draft fails the check — `pass=False` with a non-empty `mismatch` (the draft disagrees with the rules engine on one or more fields; on our run the policy id and the amount). The critic hands back the correct fields, the drafter's second attempt adopts them, and round two passes with an empty `mismatch`. The loop stops in two rounds, well under the cap of three, and `history` is the paper trail: one verdict per round, showing exactly what was wrong and when it cleared. `result` is the final decision plus a justification the model rewrote to match it.

## Judge as the critic

Swap the critic and the loop changes character. `check_decision` verifies the decision; it says nothing about whether the *justification* reads well. For that, drop the judge into the same `refine_until` in place of `check_decision`: same loop, open-ended critic. The rubric below grades form — does the justification cite a specific policy and show the arithmetic — and the drafter revises until it does.

In [ ]:
JUDGE_Q = "Does this justification cite a specific policy id and show the refund arithmetic?"
JUDGE_RUBRIC = ("Pass only if the justification names the policy id it relies on AND shows the "
                "numbers: item value, the 10% restocking fee, and the resulting refund. "
                "Appeals to fairness without the arithmetic fail.")

def draft_justification(feedback):
    msg = ("Write a one-sentence justification for a partial refund of $170.99 on an opened "
           "item returned in-window by a non-vip customer.")
    if feedback is not None:
        msg += f" A grader rejected the last one; its reasons: {feedback['reasons']}. Fix them."
    return shoplab.llm.llm(msg, model=MODEL)

def judge_critic(draft):
    return judge(JUDGE_Q, draft, JUDGE_RUBRIC)

run = refine_until(draft_justification, judge_critic, max_rounds=3)
print("stopped after", run["rounds"], "round(s)")
for i, v in enumerate(run["history"], 1):
    print(f"  round {i}: pass={v['pass']} score={v['score']} | {v['reasons'][:70]}")
print("final:", run["result"])

> **What you should see:** a first justification that skips the numbers fails the rubric; the revision that adds the item value, the fee, and the total passes — usually within two rounds. Two honest caveats separate this from the deterministic loop. First, the judge is a model: rerun the cell and the round count or the wording can move, where `check_decision` is identical every time. Second, the judge grades how the answer *reads*, not whether it is *true* — it will happily pass a justification that cites a confidently worded but nonexistent policy id, because checking that the id is real is exactly the deterministic job it cannot do. Open-ended verification buys fluency, not correctness. That is why you keep both.

## Self-Refine, in these terms

*Self-Refine: Iterative Refinement with Self-Feedback* (Madaan et al., 2023, [arXiv:2303.17651](https://arxiv.org/abs/2303.17651)) is the loop you just built, with one model playing both drafter and critic. Our version splits the roles — sometimes the critic is code, not a model — but the moving parts line up one to one.

| Paper concept | Plain English | Where it lives in code |
|---|---|---|
| Generator | produce a first attempt | `draft_fn(None)` |
| Feedback | say what is wrong with it | the verdict dict `critique_fn` returns |
| Refinement | redraft using the feedback | `draft_fn(verdict)` on the next round |
| Stop condition | quit when it is good enough, or you run out of budget | a truthy `verdict["pass"]`, capped by `max_rounds` |

## Verify the risky ten percent

Verification is not free, and the two verifiers sit at opposite ends of the cost curve. `check_decision` recomputes a spec in microseconds and bills nothing; `judge` is a network round-trip to a strong model, paid per token, and a `refine_until` loop multiplies that by its rounds. So the question is never "verify or not" but "verify *what*". The answer follows the money: reach for the deterministic check everywhere it applies — it is free, so there is no reason not to — and spend judge calls only on the open-ended slice where an error is both likely and expensive. On the ops desk that is the handful of tickets moving real money on a contestable call, not the bulk a spec settles outright. Measure the two ends so the trade-off is concrete.

In [ ]:
import time

det_t0 = time.perf_counter()
for _ in range(10000):
    shoplab.verify.check_decision(t["gold"], t, order, customer)
det_us = (time.perf_counter() - det_t0) / 10000 * 1e6

jt0 = time.perf_counter()
shoplab.verify.judge("Is a full refund justified here?",
                     "Yes, refund everything; the customer is unhappy.",
                     "Pass only if the answer applies the 10% restocking fee to an opened item.")
judge_ms = (time.perf_counter() - jt0) * 1000
row = shoplab.llm.LEDGER[-1]
print(f"check_decision: {det_us:6.1f} us/call, 0 model calls, $0")
print(f"judge:          {judge_ms:6.0f} ms/call, 1 model call, "
      f"{row['prompt_tokens']}+{row['completion_tokens']} tokens")

> **What you should see:** `check_decision` returns in single-digit microseconds and makes zero model calls; the `judge` call takes a few seconds on a cold cache and spends a few hundred tokens. That gap is several orders of magnitude, plus a bill — per verification. The lesson is not "judges are too expensive"; it is that a free, exact check should guard everything it can, leaving the judge for the cases nothing else can settle.

## Recap

| Concept | One-liner |
|---|---|
| Draft vs verify | generation and verification are separate steps; a first answer is a draft to be checked, not shipped. |
| `check_decision` | deterministic verifier — recomputes gold from `rules.decide`, flags every contradicting field, zero model calls. |
| `judge` | LLM-as-judge — grades open-ended text against a rubric, returns a JSON verdict, defaults to `STRONG_MODEL`, fails safe on junk. |
| Rubric | the judge's whole contract; vague rubrics get vague grades. |
| `refine_until` | the evaluator-optimizer loop — draft, critique, revise until `pass`, capped by `max_rounds`. |
| Critic swap | `check_decision` verifies correctness; `judge` verifies open-ended quality — same loop, different guarantee. |
| Judge caveats | nondeterministic, and it grades how an answer reads, not whether it is true. |
| Verify the risky slice | free deterministic checks everywhere they apply; paid judge calls only where an error is likely and costly. |

## Exercises

1. Write a rubric for policy-citation quality — an answer passes only if it quotes the specific clause of the policy it cites, not merely the policy id — and judge the faithful and sloppy justifications against it. Does the stricter rubric change either verdict, and can you word a justification that cites the right policy id yet still fails your rubric?
2. Make `refine_until` log its cost. Wrap `draft_decision` and `judge_critic` so each records the new rows in `shoplab.llm.LEDGER` per round, and print tokens and dollars alongside each verdict in `history`. Which loop — the deterministic critic or the judge — costs more to converge, and why?
3. Find a ticket where the two verifiers disagree: `check_decision` says the decision is wrong while `judge`, given only the justification and the cited policy, passes it. Start from a draft that cites a plausible-sounding but incorrect policy id. What does the disagreement tell you about what an open-ended judge can and cannot guarantee?

**Next up:** chapter 07 stops working with one agent and starts coordinating several — handing subtasks to specialized workers and combining what they return.